## Cell 1 — Config


In [1]:
# CELL 1 - Config
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"
print("OK Cell 1")


OK Cell 1


## Cell 2 — Clone / pull (sau Cell 1)


In [2]:
# CELL 2 - Clone or pull
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull origin {BRANCH}
%cd {WORKDIR}
assert Path("pyproject.toml").is_file(), "Chay lai Cell 1 roi Cell 2"
print("OK Cell 2 CWD=", os.getcwd())


/content/ielts-script2audio
From https://github.com/phamtu2x5/ielts-script2audio
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/ielts-script2audio
OK Cell 2 CWD= /content/ielts-script2audio


## Cell 3 — CHỈ CÀI PACKAGE (không import Orpheus)




In [3]:
# CELL 3 - INSTALL ONLY (do not import vllm / OrpheusModel here)
import sys
import subprocess

def pipi(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(">", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

print("python", sys.version)
pipi("-e", ".[dev]")
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "orpheus-tts"])

# Upstream: https://github.com/canopyai/Orpheus-TTS
pipi("orpheus-speech")
pipi("vllm==0.7.3")
pipi("--force-reinstall", "numpy==1.26.4")
pipi("--force-reinstall", "pillow==10.4.0")
pipi("soundfile")

print("OK Cell 3 pip xong.")
print(">>> BAT BUOC: Runtime -> Restart session")
print(">>> Sau do: Cell 1 -> Cell 2 -> Cell 4")


python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
> /usr/bin/python3 -m pip install -q -e .[dev]
python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
> /usr/bin/python3 -m pip install -q -e .[dev]
> /usr/bin/python3 -m pip install -q orpheus-speech
> /usr/bin/python3 -m pip install -q orpheus-speech
> /usr/bin/python3 -m pip install -q vllm==0.7.3
> /usr/bin/python3 -m pip install -q vllm==0.7.3
> /usr/bin/python3 -m pip install -q --force-reinstall numpy==1.26.4
> /usr/bin/python3 -m pip install -q --force-reinstall numpy==1.26.4
> /usr/bin/python3 -m pip install -q --force-reinstall pillow==10.4.0
> /usr/bin/python3 -m pip install -q --force-reinstall pillow==10.4.0
> /usr/bin/python3 -m pip install -q soundfile
> /usr/bin/python3 -m pip install -q soundfile
OK Cell 3 pip xong.
>>> BAT BUOC: Runtime -> Restart session
>>> Sau do: Cell 1 -> Cell 2 -> Cell 4
OK Cell 3 pip xong.
>>> BAT BUOC: Runtime -> Restart session
>>> Sau do: Cell 1 -> Cell 2 -> Cell 4


## ⏸ RESTART SESSION TẠI ĐÂY

**Runtime → Restart session**

Sau restart chạy: **Cell 1 → Cell 2 → Cell 4** (bỏ qua Cell 3 nếu cài đã xong).


## Cell 4 — Import + GPU (sau Restart)


In [3]:
# CELL 4 - IMPORT CHECK (after Restart only)
import sys
print("python", sys.version)

import numpy as np
print("numpy", np.__version__)

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Bat GPU: Runtime -> Change runtime type -> GPU")
print("device", torch.cuda.get_device_name(0))
print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

import vllm
from orpheus_tts import OrpheusModel
print("vllm", getattr(vllm, "__version__", "?"))
print("OK Cell 4: OrpheusModel import OK -> tiep Cell 5")


python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy 1.26.4
torch 2.5.1+cu124 cuda 12.4
cuda_available True
device Tesla T4
vram_gb 15.64


config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 79.5MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/snac/snac.py:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location="cpu")


vllm 0.7.3
OK Cell 4: OrpheusModel import OK -> tiep Cell 5


## Cell 5 — prepare manifest (Stage 1–2)


In [4]:
# CELL 5 - prepare
import json
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
Path("lab_audio/orpheus_part1").mkdir(parents=True, exist_ok=True)
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json
m = json.loads(Path("outputs/part1_manifest.json").read_text(encoding="utf-8"))
assert m["validation"]["valid"] is True
print("OK Cell 5 n_requests=", len(m["requests"]))
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006", "e008", "e011"}:
        print(u["event_id"], u["spoken_text"][:80])


OK Cell 5 n_requests= 15
e004 It's Sarah Thompson. That's T... H... O... M... P... S... O... N.
e006 S... W... one... A... ... one... A... A.
e008 Yes. It's zero... two... zero... ... seven... nine... four... six... ... zero...
e011 That's twelve pounds and fifty pence for the year.


## Cell 6 — Render smoke 4 đoạn (tara/leo)

**Chạy trong cùng kernel** (không `os.system` process con — tránh lỗi vLLM sau khi Cell 4 đã init CUDA).

Lần đầu load model HF sẽ **lâu** (tải weight ~3B). Smoke: `limit=4`. Full: sửa `limit=4` → `limit=None` trong code.


In [5]:
# CELL 6 - render smoke (IN-PROCESS — do not use os.system/subprocess here)
# Why: after Cell 4 imports torch/vLLM/CUDA, a new python process often fails
# with silent exit or vLLM multiprocessing errors (see Orpheus-TTS#63).

import json
import traceback
from pathlib import Path

from orpheus_tts import OrpheusModel  # already works in Cell 4
print("preflight OrpheusModel OK")

# Import lab renderer helpers from repo scripts/
import sys
sys.path.insert(0, str(Path("scripts").resolve()))
from lab_render_orpheus_from_manifest import render_manifest

OUT = Path("lab_audio/orpheus_part1")
REPORT = OUT / "lab_render_report.json"
OUT.mkdir(parents=True, exist_ok=True)

print("Starting in-process render (limit=4). First run downloads model weights (slow).")
try:
    report = render_manifest(
        Path("outputs/part1_manifest.json"),
        Path("configs/voice_maps/orpheus_en_part1.json"),
        OUT,
        use_spoken_text=True,
        limit=4,  # smoke; set to None for full 15 segments
        temperature=0.6,
        repetition_penalty=1.3,
        end_pad_ms=250,
        codes_end_pad_ms=550,
    )
except Exception as e:
    print("RENDER FAILED:", type(e).__name__, e)
    traceback.print_exc()
    # still write a tiny failure report if missing
    if not REPORT.is_file():
        REPORT.write_text(
            json.dumps(
                {
                    "backend": "orpheus",
                    "ok_count": 0,
                    "fail_count": 1,
                    "segments": [],
                    "error": f"{type(e).__name__}: {e}",
                },
                ensure_ascii=False,
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )
    raise

print("OK Cell 6 ok=", report["ok_count"], "fail=", report["fail_count"])
print("model", report.get("model_name"))
for s in report.get("segments") or []:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error") or "")
if report.get("ok_count", 0) == 0:
    raise RuntimeError("No segments rendered — see errors above")


## Cell 7 — Tracking: DISPLAY + SPOKEN + audio


In [ ]:
# CELL 7 - track
from pathlib import Path
from IPython.display import Audio, display
import json

REPORT_PATH = Path("lab_audio/orpheus_part1/lab_render_report.json")
assert REPORT_PATH.is_file(), "Chay Cell 6 truoc"
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
segments = report.get("segments") or []
audio_dir = REPORT_PATH.parent
print(report.get("transcript_id"), "n=", len(segments))
for i, seg in enumerate(segments, start=1):
    fname = seg.get("output_filename")
    wav = audio_dir / fname if fname else None
    print("=" * 72)
    print(f"[{i}/{len(segments)}] {seg.get('segment_id')} | {seg.get('speaker_id')} | {seg.get('backend_voice_id')}")
    print("DISPLAY:", seg.get("display_text", ""))
    print("SPOKEN :", seg.get("spoken_text", ""))
    if wav is not None and wav.is_file():
        display(Audio(filename=str(wav)))
    else:
        print("missing", fname)
print("OK Cell 7")


## Cell 8 — Lưu review (tuỳ chọn)


In [ ]:
# CELL 8 - reviews
from pathlib import Path
import json

report = json.loads(Path("lab_audio/orpheus_part1/lab_render_report.json").read_text(encoding="utf-8"))
reviews = {s["segment_id"]: {"content_match": "", "notes": ""} for s in report.get("segments") or []}
# reviews["seg_0001"]["content_match"] = "yes"

filled = []
missing = []
for s in report.get("segments") or []:
    sid = s["segment_id"]
    h = reviews.get(sid) or {}
    match = (h.get("content_match") or "").strip().lower()
    if match not in {"yes", "partial", "no"}:
        missing.append(sid)
    filled.append({
        "segment_id": sid,
        "display_text": s.get("display_text"),
        "spoken_text": s.get("spoken_text"),
        "backend_voice_id": s.get("backend_voice_id"),
        "output_filename": s.get("output_filename"),
        "content_match": match,
        "notes": h.get("notes", ""),
    })
out = Path("lab_audio/orpheus_part1/segment_review_filled.json")
out.write_text(json.dumps(filled, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("OK Cell 8", out, "missing", missing)


## Cell 9 — Ghép 1 file (tuỳ chọn)


In [ ]:
# CELL 9 - concat
import json
from pathlib import Path
from IPython.display import Audio, display

assert Path("lab_audio/orpheus_part1/lab_render_report.json").is_file()
!python scripts/lab_concat_segment_wavs.py --report lab_audio/orpheus_part1/lab_render_report.json --out lab_audio/orpheus_part1/part1_full.wav --gap-mode adaptive --same-speaker-gap-ms 220 --turn-gap-ms 520
full = Path("lab_audio/orpheus_part1/part1_full.wav")
print(json.loads(Path("lab_audio/orpheus_part1/part1_full.concat.json").read_text()).get("duration_sec"), "sec")
if full.is_file():
    display(Audio(filename=str(full)))
print("OK Cell 9")


## Cell 10 — Survey nhiều giọng (tuỳ chọn, lâu)


In [ ]:
# CELL 10 - survey
from pathlib import Path
from IPython.display import Audio, display
import json
from collections import defaultdict

from orpheus_tts import OrpheusModel
print("preflight OK")

!python scripts/lab_survey_orpheus_voices.py --manifest outputs/part1_manifest.json --inventory configs/voice_maps/orpheus_voice_inventory.json --preset en_shortlist --out-dir lab_audio/orpheus_voice_survey --event-ids e004,e006,e008,e011 --end-pad-ms 450

rp = Path("lab_audio/orpheus_voice_survey/voice_survey_report.json")
assert rp.is_file(), "Survey fail"
report = json.loads(rp.read_text(encoding="utf-8"))
print("OK Cell 10", report.get("ok_count"), report.get("fail_count"), report.get("voices"))
by = defaultdict(list)
for r in report.get("renders") or []:
    by[r.get("event_id")].append(r)
for eid, items in by.items():
    items = sorted(items, key=lambda x: x.get("voice_id") or "")
    print("=" * 72, "LINE", eid)
    if items:
        print("DISPLAY", items[0].get("display_text"))
        print("SPOKEN ", items[0].get("spoken_text"))
    for it in items:
        print("---", it.get("voice_id"), it.get("status"))
        p = Path("lab_audio/orpheus_voice_survey") / it["output_filename"]
        if p.is_file():
            display(Audio(filename=str(p)))


Xong. Verdict lab only — `not_selected`. Không sửa script.
